1. Persiapan dan Instalasi Library

In [3]:
# Install library yang dibutuhkan
!pip install tensorflow pandas numpy scikit-learn sentence-transformers -q

import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sentence_transformers import SentenceTransformer

# Mount Google Drive untuk mengakses dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


2. Memuat dan Memproses Data (Data Preprocessing)

In [4]:
# Tentukan path dataset
movies_path = '/content/drive/MyDrive/dataset sistem rekomendasi/movies.csv'
ratings_path = '/content/drive/MyDrive/dataset sistem rekomendasi/ratings.csv'

# Memuat dataset
print("Memuat dataset dari Google Drive...")
movies = pd.read_csv(movies_path)
ratings = pd.read_csv(ratings_path)

# Gabungkan data untuk mendapatkan metadata teks pada setiap rating
df = ratings.merge(movies, on='movieId')

# 1. Encoding User ID dan Item ID ke rentang kontinu
user_ids = df['userId'].astype('category').cat.codes.values
item_ids = df['movieId'].astype('category').cat.codes.values

df['user_encoded'] = user_ids
df['item_encoded'] = item_ids

num_users = df['user_encoded'].nunique()
num_items = df['item_encoded'].nunique()

print(f"Jumlah User: {num_users}, Jumlah Item: {num_items}")

# 2. Siapkan label (misal: memprediksi probabilitas interaksi positif / rating >= 3.5)
df['interaction'] = (df['rating'] >= 3.5).astype(int)

Memuat dataset dari Google Drive...
Jumlah User: 1819, Jumlah Item: 14018


3. Ekstraksi Fitur Teks dengan BERT (Feature Engineering)

In [5]:
# Inisialisasi pre-trained model BERT yang ringan dan cepat
print("Memuat model BERT (all-MiniLM-L6-v2)...")
bert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Membuat fitur teks dengan menggabungkan Judul dan Genre
# Memastikan penanganan string kosong atau null
movies['title'] = movies['title'].fillna('')
movies['genres'] = movies['genres'].fillna('').str.replace('|', ' ')
movies['text_feature'] = movies['title'] + " " + movies['genres']

# Mengekstrak embedding untuk setiap film unik
print("Mengekstrak embedding teks...")
item_text_embeddings = bert_model.encode(movies['text_feature'].tolist(), show_progress_bar=True)

# Buat dictionary/mapping dari item_encoded ke vektor BERT
item_id_to_encoded = dict(zip(movies['movieId'], movies['movieId'].astype('category').cat.codes))

# Buat matriks embedding teks yang sejajar dengan indeks item_encoded
embedding_dim_bert = item_text_embeddings.shape[1] # Umumnya 384 dimensi untuk MiniLM
bert_embedding_matrix = np.zeros((num_items, embedding_dim_bert))

for idx, row in movies.iterrows():
    encoded_id = item_id_to_encoded.get(row['movieId'])
    if pd.notna(encoded_id) and encoded_id < num_items:
        bert_embedding_matrix[int(encoded_id)] = item_text_embeddings[idx]

print(f"Bentuk Matriks BERT: {bert_embedding_matrix.shape}")

Memuat model BERT (all-MiniLM-L6-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Mengekstrak embedding teks...


Batches:   0%|          | 0/1951 [00:00<?, ?it/s]

Bentuk Matriks BERT: (14018, 384)


4. Membangun Arsitektur Model BERT + NCF

In [6]:
# Hyperparameters
embedding_size = 50
hidden_units = [128, 64, 32]
dropout_rate = 0.2

# --- INPUT LAYERS ---
user_input = Input(shape=(1,), name='user_input')
item_input = Input(shape=(1,), name='item_input')
bert_input = Input(shape=(embedding_dim_bert,), name='bert_input')

# --- EMBEDDING LAYERS ---
# Collaborative Filtering Embeddings
user_embedding = Embedding(input_dim=num_users, output_dim=embedding_size, name='user_embedding')(user_input)
item_embedding = Embedding(input_dim=num_items, output_dim=embedding_size, name='item_embedding')(item_input)

# Flatten Embeddings
user_flatten = Flatten()(user_embedding)
item_flatten = Flatten()(item_embedding)

# Pemrosesan fitur BERT (Mereduksi dimensi teks agar lebih ringkas)
bert_dense = Dense(64, activation='relu', name='bert_dense')(bert_input)

# --- CONCATENATION ---
# Menggabungkan fitur CF dan fitur Teks
concat = Concatenate()([user_flatten, item_flatten, bert_dense])

# --- MULTI-LAYER PERCEPTRON (MLP) ---
mlp = Dense(hidden_units[0], activation='relu')(concat)
mlp = Dropout(dropout_rate)(mlp)
mlp = Dense(hidden_units[1], activation='relu')(mlp)
mlp = Dropout(dropout_rate)(mlp)
mlp = Dense(hidden_units[2], activation='relu')(mlp)

# --- OUTPUT LAYER ---
output = Dense(1, activation='sigmoid', name='prediction')(mlp)

# Compile Model
model = Model(inputs=[user_input, item_input, bert_input], outputs=output)
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 50)     │     90,950 │ user_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_embedding      │ (None, 1, 50)     │    700,900 │ item_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_input          │ (None, 384)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 50)        │          0 │ user_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 50)        │          0 │ item_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_dense (Dense)  │ (None, 64)        │     24,640 │ bert_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 164)       │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0],  │
│                     │                   │            │ bert_dense[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     21,120 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ prediction (Dense)  │ (None, 1)         │         33 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 847,979 (3.23 MB)

 Trainable params: 847,979 (3.23 MB)

 Non-trainable params: 0 (0.00 B)

5. Persiapan Data Latih dan Pelatihan Model (Training)

In [7]:
# Siapkan input arrays untuk training
X_users = df['user_encoded'].values
X_items = df['item_encoded'].values

# Petakan vektor BERT ke setiap baris transaksi berdasarkan item_encoded
X_bert = np.array([bert_embedding_matrix[i] for i in X_items])
y = df['interaction'].values

# Train-Test Split (80% Train, 20% Test)
print("Memisahkan data latih dan data uji...")
X_u_train, X_u_test, X_i_train, X_i_test, X_b_train, X_b_test, y_train, y_test = train_test_split(
    X_users, X_items, X_bert, y, test_size=0.2, random_state=42
)

# Proses Training
print("Memulai proses training...")
history = model.fit(
    x=[X_u_train, X_i_train, X_b_train],
    y=y_train,
    batch_size=256,
    epochs=10,
    validation_split=0.1
)

Memisahkan data latih dan data uji...
Memulai proses training...
Epoch 1/10
723/723 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - accuracy: 0.7116 - loss: 0.5560 - val_accuracy: 0.7363 - val_loss: 0.5261
Epoch 2/10
723/723 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.7541 - loss: 0.5014 - val_accuracy: 0.7398 - val_loss: 0.5224
Epoch 3/10
723/723 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.7669 - loss: 0.4791 - val_accuracy: 0.7389 - val_loss: 0.5276
Epoch 4/10
723/723 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.7777 - loss: 0.4605 - val_accuracy: 0.7371 - val_loss: 0.5396
Epoch 5/10
723/723 ━━━━━━━━━━━━━━━━━━━━ 12s 14ms/step - accuracy: 0.7896 - loss: 0.4406 - val_accuracy: 0.7353 - val_loss: 0.5532
Epoch 6/10
723/723 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.8001 - loss: 0.4198 - val_accuracy: 0.7338 - val_loss: 0.5737
Epoch 7/10
723/723 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.8121 - loss: 0.3985 - val_accuracy: 0.7273 - val_loss: 0.6000
Epoch 8/10
723/723 ━━━━━━━

6. Evaluasi dan Fungsi Rekomendasi

In [8]:
# Evaluasi pada data uji (Test Set)
print("\nMengevaluasi model pada data uji...")
loss, accuracy = model.evaluate([X_u_test, X_i_test, X_b_test], y_test)
print(f'Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}')

# Fungsi untuk memberikan Top-N Rekomendasi
def recommend_movies(user_id_encoded, top_n=5):
    # Validasi apakah user valid
    if user_id_encoded >= num_users or user_id_encoded < 0:
        print(f"Error: user_id_encoded harus antara 0 dan {num_users-1}")
        return

    # Buat array item untuk semua film yang ada
    all_items = np.arange(num_items)
    user_array = np.full(num_items, user_id_encoded)

    # Ambil fitur BERT untuk semua item
    all_bert_features = bert_embedding_matrix

    # Prediksi skor probabilitas
    predictions = model.predict([user_array, all_items, all_bert_features], batch_size=512, verbose=0).flatten()

    # Ambil indeks item dengan skor tertinggi
    top_indices = predictions.argsort()[-top_n:][::-1]

    print(f"\nRekomendasi Top-{top_n} untuk User Index {user_id_encoded}:")
    for idx in top_indices:
        # Kembalikan ke ID asli dan cari nama filmnya
        try:
            original_movie_id = list(item_id_to_encoded.keys())[list(item_id_to_encoded.values()).index(idx)]
            movie_title = movies[movies['movieId'] == original_movie_id]['title'].values[0]
            score = predictions[idx]
            print(f"- {movie_title} (Skor Prediksi: {score:.4f})")
        except ValueError:
            continue

# Uji sistem rekomendasi untuk user index 0
recommend_movies(user_id_encoded=15, top_n=5)


Mengevaluasi model pada data uji...
1606/1606 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.7213 - loss: 0.7186
Test Loss: 0.7186, Test Accuracy: 0.7213

Rekomendasi Top-5 untuk User Index 15:
- Big Night (1996) (Skor Prediksi: 1.0000)
- Bad Boys II (2003) (Skor Prediksi: 1.0000)
- Radio Days (1987) (Skor Prediksi: 1.0000)
- Eye of the Tiger (1986) (Skor Prediksi: 1.0000)
- Enemy at the Gates (2001) (Skor Prediksi: 1.0000)


**UAS PRAKTEK SISTEM REKOMENDASI**

menyimpan model yang sudah ditraining di Google Colab

In [11]:
import pickle

# 1. Simpan Model NCF
model.save('/content/drive/MyDrive/dataset sistem rekomendasi/SR/ncf_bert_model.h5')

# 2. Simpan Vektor BERT Embedding
np.save('/content/drive/MyDrive/dataset sistem rekomendasi/SR/bert_embeddings.npy', bert_embedding_matrix)

# 3. Simpan Mapping Item (Kamus ID)
with open('/content/drive/MyDrive/dataset sistem rekomendasi/SR/item_mapping.pkl', 'wb') as f:
    pickle.dump(item_id_to_encoded, f)

# 4. Simpan Metadata Film (untuk menampilkan judul di API)
movies[['movieId', 'title']].to_csv('/content/drive/MyDrive/dataset sistem rekomendasi/SR/movies_meta.csv', index=False)

print("Semua file model berhasil disimpan di Google Drive!")

Semua file model berhasil disimpan di Google Drive!


In [ ]:
Menyimpan Kode API (api.py)

In [25]:
%%writefile api.py
from fastapi import FastAPI, HTTPException
import numpy as np
import pandas as pd
import tensorflow as tf
import pickle

app = FastAPI(title="Movie Recommendation API")

# --- LOAD MODEL (Langsung ambil dari Drive Anda) ---
model = tf.keras.models.load_model('/content/drive/MyDrive/dataset sistem rekomendasi/SR/ncf_bert_model.h5')
bert_embeddings = np.load('/content/drive/MyDrive/dataset sistem rekomendasi/SR/bert_embeddings.npy')
with open('/content/drive/MyDrive/dataset sistem rekomendasi/SR/item_mapping.pkl', 'rb') as f:
    item_mapping = pickle.load(f)
movies_meta = pd.read_csv('/content/drive/MyDrive/dataset sistem rekomendasi/SR/movies_meta.csv')

num_items = bert_embeddings.shape[0]

@app.get("/")
def read_root():
    return {"message": "API Sistem Rekomendasi Aktif!"}

@app.get("/recommend/{user_id}")
def get_recommendations(user_id: int, top_n: int = 5):
    if user_id < 0 or user_id > 1819:
        raise HTTPException(status_code=404, detail="User ID tidak valid (Gunakan 0-1819).")

    all_items = np.arange(num_items)
    user_array = np.full(num_items, user_id)

    # Prediksi menggunakan model NCF
    predictions = model.predict([user_array, all_items, bert_embeddings], batch_size=512, verbose=0).flatten()
    top_indices = predictions.argsort()[-top_n:][::-1]

    recommendations = []
    for idx in top_indices:
        try:
            original_movie_id = list(item_mapping.keys())[list(item_mapping.values()).index(idx)]
            movie_title = movies_meta[movies_meta['movieId'] == original_movie_id]['title'].values[0]
            score = float(predictions[idx])

            recommendations.append({
                "title": movie_title,
                "confidence_score": round(score, 4)
            })
        except:
            continue

    return {"user_id": user_id, "recommendations": recommendations}

Overwriting api.py


Menyalakan API di Background
Kita akan menggunakan modul subprocess milik Python untuk menyalakan server Uvicorn tanpa membuat Colab terblokir (loading terus-menerus).

In [29]:
import subprocess
import time

# Pastikan uvicorn sudah terinstal
!pip install fastapi uvicorn -q

# Matikan proses uvicorn sebelumnya (jika ada) untuk mencegah bentrok port
!pkill uvicorn

print("⏳ Sedang menyalakan API Server di background...")
# Jalankan uvicorn di background pada port 8000
process = subprocess.Popen(["uvicorn", "api:app", "--host", "127.0.0.1", "--port", "8000"])

# Tunggu 5 detik untuk memastikan server sudah sepenuhnya hidup
time.sleep(5)
print("✅ API Server sudah menyala dan siap menerima request di http://127.0.0.1:8000")

⏳ Sedang menyalakan API Server di background...
✅ API Server sudah menyala dan siap menerima request di http://127.0.0.1:8000


Antarmuka CLI (Aplikasi Pengguna)
Ini adalah kode yang bertindak sebagai "Aplikasi Client". Saat sel ini dijalankan, akan muncul kolom input (seperti terminal)

In [ ]:
import requests

API_URL = "http://127.0.0.1:8000/recommend"

def main():
    print("="*55)
    print("🎬 APLIKASI SISTEM REKOMENDASI FILM (CLI) 🎬")
    print("="*55)
    print("Ketik 'exit' atau 'keluar' untuk menghentikan aplikasi.\n")

    while True:
        # Colab akan memunculkan box input interaktif di sini
        user_input = input("➡️ Masukkan User ID (contoh: 0, 15, 100): ")

        if user_input.lower() in ['exit', 'keluar']:
            print("\nTerima kasih telah menggunakan sistem rekomendasi!")
            break

        if not user_input.isdigit():
            print("⚠️ Harap masukkan angka yang valid!\n")
            continue

        user_id = int(user_input)
        top_n = 5

        print(f"⏳ Mengambil data dari API untuk User {user_id}...")

        try:
            # Memanggil API yang menyala di background Colab
            response = requests.get(f"{API_URL}/{user_id}?top_n={top_n}")

            if response.status_code == 200:
                data = response.json()
                recs = data.get("recommendations", [])

                print(f"\n✅ REKOMENDASI TOP-{top_n} UNTUK USER {user_id}:")
                for i, rec in enumerate(recs, 1):
                    print(f"   {i}. {rec['title']} (Skor Probabilitas: {rec['confidence_score']})")
                print("-" * 55 + "\n")

            elif response.status_code == 404:
                print(f"❌ Error API: {response.json()['detail']}\n")
            else:
                print(f"❌ Error API: Terjadi kesalahan server (Code: {response.status_code})\n")

        except requests.exceptions.ConnectionError:
            print("❌ Error: Tidak dapat terhubung ke API. Pastikan Sel 2 (Server) sudah dijalankan!\n")

if __name__ == "__main__":
    main()

🎬 APLIKASI SISTEM REKOMENDASI FILM (CLI) 🎬
Ketik 'exit' atau 'keluar' untuk menghentikan aplikasi.

➡️ Masukkan User ID (contoh: 0, 15, 100): 0
⏳ Mengambil data dari API untuk User 0...

✅ REKOMENDASI TOP-5 UNTUK USER 0:
   1. Mummy, The (1999) (Skor Probabilitas: 1.0)
   2. I'm All Right Jack (1959) (Skor Probabilitas: 1.0)
   3. Sorority House Massacre II (1990) (Skor Probabilitas: 1.0)
   4. Radio Days (1987) (Skor Probabilitas: 1.0)
   5. CSNY Déjà Vu (2008) (Skor Probabilitas: 1.0)
-------------------------------------------------------

➡️ Masukkan User ID (contoh: 0, 15, 100): 15
⏳ Mengambil data dari API untuk User 15...

✅ REKOMENDASI TOP-5 UNTUK USER 15:
   1. Big Night (1996) (Skor Probabilitas: 1.0)
   2. Bad Boys II (2003) (Skor Probabilitas: 1.0)
   3. Radio Days (1987) (Skor Probabilitas: 1.0)
   4. Eye of the Tiger (1986) (Skor Probabilitas: 1.0)
   5. Enemy at the Gates (2001) (Skor Probabilitas: 1.0)
-------------------------------------------------------

